In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 0. 초기 설정

In [13]:
import pandas as pd
import json
import os
import ast
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
LLM_PROVIDER    = "openai"
OPENAI_MODEL    = "gpt-4o-mini" # "gpt-5-mini"
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")

# 1. 유저 프롬프트 생성(제품정보+규칙)

In [19]:
# 1. 제품정보 csv 파일을 읽어 프롬프트 생성
try:
    product_df = pd.read_csv('data/product_info_v2.csv')

    # 제품 정보 프롬프트 생성
    product_list_prompt = "15개의 제품 정보는 아래와 같아\n"
    product_list = []
    for _, row in product_df.iterrows():
        product_info_str = f"- 제품명: {row['product_name']}\n  제품 특징: {row['product_feature']}\n"
        product_list_prompt += product_info_str
        product_list.append(row['product_name'])
    
except FileNotFoundError:
    print("product_info.csv 파일을 찾을 수 없습니다. 파일 경로를 확인해주세요.")
    product_list_prompt = ""
    
# 2. 규칙을 추가하여 user 프롬프트 생성
constaint_prompt = '''규칙:
- 각 제품에 대해 월간 구매전환확률 p(0~100)과 월간 예상구매수 q(>=0)를 추정하라.
- 정보가 없으면 중립값을 사용하고 추정하되, 임의 상상/할루시네이션 금지.
- 논리는 출력하지 말고 결과 값만 반환.
- 스케일 가이드(선형 아님): 편의성↑, 가격↑(=비싸짐)은 p↓, 페르소나의 선호/용도 일치 시 p,q↑.
- month_purchase = round(p*q/100,2)
- 제품목록 순서대로 month_purchase 숫자만 출력. 아래 스키마 참고

출력 리스트 스키마(숫자만):
[ "month_purchase_1", "month_purchase_2", "month_purchase_3", ... , "month_purchase_14", "month_purchase_15"]
'''
user_prompt = f"{product_list_prompt}\n{constaint_prompt}"
print(user_prompt)

15개의 제품 정보는 아래와 같아
- 제품명: 덴마크 하이그릭요거트 400g
  제품 특징: 이중 유청분리 공범, 꾸덕한 질감, 고소함, 호상(떠먹는) 발효유, 고단백, 아연, 칼슘, 6-7월 TV/Youtube/SNS 광고 진행, 6-8월 수도권 중심 아파트 엘리베이터 광고, 광고모델: 일반인
- 제품명: 동원맛참 고소참기름 135g
  제품 특징: 광고모델: 안유진, 참기름, 단백질, 셀레늄, 고소
- 제품명: 동원맛참 고소참기름 90g
  제품 특징: 광고모델: 안유진, 참기름, 단백질, 셀레늄, 고소
- 제품명: 동원맛참 매콤참기름 135g
  제품 특징: 광고모델: 안유진, 참기름, 단백질, 셀레늄, 매콤
- 제품명: 동원맛참 매콤참기름 90g
  제품 특징: 광고모델: 안유진, 참기름, 단백질, 셀레늄, 매콤
- 제품명: 동원참치액 순 500g
  제품 특징: 훈연참치추출물 80%, 참기엑기스, 참치명가, 직접 잡은 참치, 가쓰오엑기스, 표고버섯, 무, 감초, 마늘
- 제품명: 동원참치액 순 900g
  제품 특징: 훈연참치추출물 80%, 참기엑기스, 참치명가, 직접 잡은 참치, 가쓰오엑기스, 표고버섯, 무, 감초, 마늘
- 제품명: 동원참치액 진 500g
  제품 특징: 훈연참치추출물 80%, 참기엑기스, 참치명가, 직접 잡은 참치, 가쓰오엑기스, 표고버섯, 무, 감초, 마늘
- 제품명: 동원참치액 진 900g
  제품 특징: 훈연참치추출물 80%, 참기엑기스, 참치명가, 직접 잡은 참치, 가쓰오엑기스, 표고버섯, 무, 감초, 마늘
- 제품명: 리챔 오믈레햄 200g
  제품 특징: 오믈렛(Omelet)과 햄(Ham)의 합성어, 저나트륨, 내열성 케첩 소스
- 제품명: 리챔 오믈레햄 340g
  제품 특징: 오믈렛(Omelet)과 햄(Ham)의 합성어, 저나트륨, 내열성 케첩 소스
- 제품명: 소화가 잘되는 우유로 만든 바닐라라떼 250mL
  제품 특징: 락토프리, 저온효소 처리 기술, 유당 ZERO, 저당 트렌드, 신선한 1등급 원유, SNS 

In [20]:
product_list

['덴마크 하이그릭요거트 400g',
 '동원맛참 고소참기름 135g',
 '동원맛참 고소참기름 90g',
 '동원맛참 매콤참기름 135g',
 '동원맛참 매콤참기름 90g',
 '동원참치액 순 500g',
 '동원참치액 순 900g',
 '동원참치액 진 500g',
 '동원참치액 진 900g',
 '리챔 오믈레햄 200g',
 '리챔 오믈레햄 340g',
 '소화가 잘되는 우유로 만든 바닐라라떼 250mL',
 '소화가 잘되는 우유로 만든 카페라떼 250mL',
 '프리미엄 동원참치액 500g',
 '프리미엄 동원참치액 900g']

# 2. 반복 질문

In [11]:
# 배치 및 분할 작업을 위한 변수
batch_start = 11
batch_end = 20

In [15]:
%%time
# 1. persona_core.json 파일에서 페르소나 정보 불러오기
rows = []

with open('json/persona_core_summary_fixed.json', 'r', encoding='utf-8') as f:
    persona_data = json.load(f)

# 페르소나 정보 추출 (batch_start ~ batch_end)
for i in range(batch_start, batch_end):
    persona_summary = persona_data[i]['summary']

    # 페르소나 프롬프트 생성
    persona_prompt = f"너는 다음과 같은 페르소나를 가진 사람이야: '{persona_summary}'"
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(model=OPENAI_MODEL,
                                          messages=[
                                        {"role": "system", "content": f"{persona_prompt}. 단, 응답은 반드시 띄어쓰기 및 내려쓰기 없이 list로만 출력해."},
                                        {"role": "user", "content": user_prompt },
                                        ],
                                        )
    raw = resp.choices[0].message.content
    row = list(map(float, ast.literal_eval(raw)))  # 문자열 → 리스트(float)
    rows.append(row)
    
df = pd.DataFrame(rows)
df

CPU times: total: 3.36 s
Wall time: 4min 18s


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0.90,0.03,0.06,0.01,0.02,0.05,0.01,0.05,0.01,0.40,0.35,2.40,2.40,0.06,0.01
1,0.10,0.05,0.07,0.04,0.07,0.10,0.05,0.11,0.05,0.78,0.33,0.15,0.15,0.06,0.03
2,0.23,0.06,0.12,0.04,0.06,0.30,0.44,0.29,0.42,0.42,0.59,0.66,0.66,0.29,0.50
3,1.20,0.06,0.08,0.03,0.03,0.03,0.01,0.01,0.01,0.20,0.14,0.75,0.83,0.01,0.00
4,0.35,0.12,0.08,0.06,0.04,0.20,0.18,0.18,0.15,0.30,0.22,0.20,0.20,0.28,0.20
5,0.15,0.02,0.03,0.01,0.01,0.04,0.06,0.04,0.07,0.54,0.65,0.06,0.10,0.02,0.05
6,0.18,0.30,0.20,0.18,0.15,0.42,0.26,0.43,0.27,0.20,0.32,0.11,0.11,0.48,0.30
7,0.65,0.08,0.04,0.04,0.02,0.09,0.12,0.07,0.10,0.06,0.08,0.30,0.29,0.14,0.20
8,0.75,0.84,0.98,0.55,0.60,0.48,0.33,0.56,0.34,0.78,0.48,0.55,0.55,0.30,0.18
9,1.05,0.15,0.16,0.09,0.11,0.23,0.07,0.16,0.08,0.50,0.27,1.00,1.00,0.30,0.16


In [21]:
df_sum = df.sum()*200
df_full = df_sum.to_frame(name='month_purchase')
df_full.index = product_list
df_full

,month_purchase
덴마크 하이그릭요거트 400g,1112.0
동원맛참 고소참기름 135g,342.0
동원맛참 고소참기름 90g,364.0
동원맛참 매콤참기름 135g,210.0
동원맛참 매콤참기름 90g,222.0
동원참치액 순 500g,388.0
동원참치액 순 900g,306.0
동원참치액 진 500g,380.0
동원참치액 진 900g,300.0
리챔 오믈레햄 200g,836.0


In [23]:
df_sum_0_9 = df.sum()
df_sum_0_9

0     5.56
1     1.71
2     1.82
3     1.05
4     1.11
5     1.94
6     1.53
7     1.90
8     1.50
9     4.18
10    3.43
11    6.18
12    6.29
13    1.94
14    1.63
dtype: float64